## Catálogo hospital -> estado

In [1]:
import sys
from pathlib import Path
import pandas as pd
import re

ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT / 'src'))
DATA_DIR =  ROOT / 'data'

## 1. Excluir encabezado y limpiar sufijos de número romano

In [2]:
def _limpiar_nombre_hospital(nombre: str) -> str: 
    # quita sufijo de número romano al final (con o sin "RM"), tolera 0 o más espacios
    return re.sub(r"\s*[IVXivx]+\s*(RM|-A)?\s*$", "", nombre.strip()).strip()

In [3]:
_EXCLUIR = {"ESCALONES SANITARIOS", "RECETAS EXPEDIDAS", "TOTAL", "TOTAL GENERAL"}

xl = pd.ExcelFile(DATA_DIR / "SEDENA_2021_ABRIL2024_ANEXO FOLIO 330026424001543.xlsx")
hospitales = set()
for hoja in ["2021", "2022", "2023"]:
    df = xl.parse(hoja, header=None)
    col_hospital = 3 if hoja == "2021" else 1
    valores = df[col_hospital].dropna()
    valores = valores[valores.apply(lambda x: isinstance(x, str))]
    for v in valores:
        limpio = _limpiar_nombre_hospital(v)
        if limpio.upper() not in _EXCLUIR and limpio:
            hospitales.add(limpio)

for h in sorted(hospitales):
    print(h)


CENTRO DE REHABILITACIÓN INFANTIL
CHEMP. LOS PINOS, D.F. H.M.Z. CONSTITUYENTES
H. M. E. DE LA MUJ. Y NEONATOLOGIA.
H. M. R. CHILPANCINGO, GRO.
H. M. R. DE CHIHUAHUA, CHIH.
H. M. R. DE EL CIPRES.
H. M. R. HERMOSILLO, SON.
H. M. R. IRAPUATO, GTO.
H. M. R. LA PAZ, B.C.
H. M. R. MAZATLAN, SIN.
H. M. R. MERIDA, YUC.
H. M. R. PUEBLA, PUE.
H. M. R. SAN LUIS POTOSI, S.L.P.
H. M. R. TAMPICO, TAMPS.
H. M. R. TORREON, COAH.
H. M. R. TUXPAN, VER.
H. M. R. TUXTLA GUTIERREZ, CHIS.
H. M. Y UNIDAD. ESP. MED. DE GUAD. JAL.
H. M. Z. 5 DE MAYO, DGO.
H. M. Z. APATZINGAN, MICH.
H. M. Z. CHETUL, Q. ROO.
H. M. Z. CHETUMAL, Q. ROO.
H. M. Z. CPO. MIL. No. 1 -A, CD. MÉX.
H. M. Z. CUERNAVACA, MOR.
H. M. Z. GUADALUPE, ZAC.
H. M. Z. IXCOTEL, OAX.
H. M. Z. IXTEPEC, OAX.
H. M. Z. LA BOTICARIA, VER.
H. M. Z. MEXICALI, B.C.
H. M. Z. SAN MIGUEL DE LOS JAHUYES, MEX.
H. M. Z. SANTA GERTRUDIS, CHIH.
H. M. Z. SANTA LUCIA, MEX.
H. M. Z. SANTA MARIA RAYON, MEX.
H. M. Z. TEMAMATLA, MEX.
H. M. Z. VILLAHERMOSA, TAB.
H. M. Z. ZA

## 2. Extrae el estado por abreviatura embedida (regex), en vez de mapear a mano 

In [4]:
_ABREV_ESTADO = {
    "SIN.": "SINALOA", "CHIH.": "CHIHUAHUA", "JAL.": "JALISCO", "GTO.": "GUANAJUATO",
    "MEX.": "MÉXICO", "CD. MÉX.": "CDMX", "D.F.": "CDMX", "MOR.": "MORELOS",
    "ZAC.": "ZACATECAS", "OAX.": "OAXACA", "VER.": "VERACRUZ", "TAB.": "TABASCO",
    "CHIS.": "CHIAPAS", "TAMPS.": "TAMAULIPAS", "S.L.P.": "SAN LUIS POTOSI",
    "GRO.": "GUERRERO", "DGO.": "DURANGO", "MICH.": "MICHOACÁN",
    "Q. ROO.": "QUINTANA ROO", "B.C.": "BAJA CALIFORNIA", "SON.": "SONORA",
    "COAH.": "COAHUILA", "YUC": "YUCATÁN", "PUE": "PUEBLA"
}

def _extraer_estado(nombre: str) -> str | None:
    for abrev, estado in _ABREV_ESTADO.items():
        if nombre.upper().rstrip(".").endswith(abrev.rstrip(".")):
            return estado
    return None


In [5]:
sin_estado = [h for h in sorted(hospitales) if _extraer_estado(h) is None]
print(sin_estado)

['CENTRO DE REHABILITACIÓN INFANTIL', 'CHEMP. LOS PINOS, D.F. H.M.Z. CONSTITUYENTES', 'H. M. E. DE LA MUJ. Y NEONATOLOGIA.', 'H. M. R. DE EL CIPRES.', 'H.M.Z CONTITUYENTES', 'HOSP. CENTRAL MILITAR', 'HOSP.. MIL. DE ESPECIALIDADES DE  MONTERREY', 'U. M. C. E. DE LA S.D.N.', 'U. M. C. E. TECAMACHALCO, D.F.  (ORIENTAL, PUEBLA).', 'U.M.C.E. LEONES, TACUBA', 'UNIDAD ESP. MEDICAS.', 'UNIDAD ESP. ODONTOLOGICAS.']


In [6]:
_MAPEO_MANUAL = {
    "CENTRO DE REHABILITACIÓN INFANTIL": "CDMX",
    "H. M. E. DE LA MUJ. Y NEONATOLOGIA.": "CDMX",
    "H.M.Z CONTITUYENTES": "CDMX", # typo de "CONSTITUYENTES", CDMX
    "HOSP. CENTRAL MILITAR": "CDMX",
    "U. M. C. E. DE LA S.D.N.": "CDMX",
    "U.M.C.E. LEONES, TACUBA": "CDMX", # Tacuba es colonia de CDMX
    "UNIDAD ESP. MEDICAS.": "CDMX",
    "UNIDAD ESP. ODONTOLOGICAS.": "CDMX",
    "CHEMP. LOS PINOS, D.F. H.M.Z. CONSTITUYENTES": "CDMX",  # captura sucia: dos nombres pegados, ambos CDMX

    "HOSP.. MIL. DE ESPECIALIDADES DE  MONTERREY": "NUEVO LEÓN", # ciudad completa, no abreviatura
    "H. M. R. DE EL CIPRES.": "BAJA CALIFORNIA", # Hospital de El Ciprés, Ensenada B.C.

    # el nombre dice "D.F." pero el paréntesis aclara que es Puebla:
    "U. M. C. E. TECAMACHALCO, D.F.  (ORIENTAL, PUEBLA).": "PUEBLA",
}


In [7]:
def estado_de_hospital(nombre: str) -> str | None:
    limpio = _limpiar_nombre_hospital(nombre)
    return _extraer_estado(limpio) or _MAPEO_MANUAL.get(limpio)

In [8]:
sin_estado = [h for h in sorted(hospitales) if estado_de_hospital(h) is None]
print(sin_estado)

[]


In [9]:
_SEDENA_LAYOUT = {
    "2021": {"col_hospital": 3, "col_expedidas": 4, "col_surtidas": 6},
    "2022": {"col_hospital": 1, "col_expedidas": 2, "col_surtidas": 3},
    "2023": {"col_hospital": 1, "col_expedidas": 2, "col_surtidas": 3},
}

def _parse_sedena_sheet(xl, hoja: str) -> pd.DataFrame:
    cfg = _SEDENA_LAYOUT[hoja]
    raw = xl.parse(hoja, header = None)
    data = raw.copy()
    data["hospital_raw"] = data[cfg["col_hospital"]]
    data = data[data["hospital_raw"].apply(lambda x: isinstance(x, str))].copy()
    data["hospital"] = data["hospital_raw"].apply(_limpiar_nombre_hospital)
    data = data[~data["hospital"].str.upper().isin(_EXCLUIR) & (data["hospital"] != "")]
    data["total"] = pd.to_numeric(data[cfg["col_expedidas"]], errors = "coerce")
    data["surtidas"] = pd.to_numeric(data[cfg["col_surtidas"]], errors = "coerce")
    data["anio"] = int(hoja)
    return data[["hospital", "anio", "surtidas", "total"]].dropna()

partes = [_parse_sedena_sheet(xl, h) for h in ["2021", "2022", "2023"]]
df_sedena = pd.concat(partes, ignore_index=True)
df_sedena["estado"] = df_sedena["hospital"].apply(estado_de_hospital)


In [10]:
faltantes_2021 = df_sedena[df_sedena["estado"].isna()]["hospital"].unique()
print(faltantes_2021)

[]


In [11]:
# validación: ningún estado nulo, y totales razonables
assert df_sedena["estado"].notna().all()
print(df_sedena.groupby(["estado", "anio"])[["surtidas", "total"]].sum())

                      surtidas   total
estado          anio                  
BAJA CALIFORNIA 2021     53262   60168
                2022     32605   43424
                2023     57006   74541
CDMX            2021    419797  484408
                2022    441332  506661
...                        ...     ...
YUCATÁN         2022     30808   43881
                2023     29422   42809
ZACATECAS       2021     15850   28677
                2022      9446   11337
                2023     12070   20086

[72 rows x 2 columns]


In [12]:
estados_base = set(pd.read_csv(DATA_DIR / "base_modelo_2019_2023.csv")["estado"].unique())
estados_sedena = set(df_sedena["estado"].unique())
print("En SEDENA pero no en base:", estados_sedena - estados_base)
print("En base pero no en SEDENA:", estados_base - estados_sedena)


En SEDENA pero no en base: set()
En base pero no en SEDENA: {'NAYARIT', 'TLAXCALA', 'BAJA CALIFORNIA SUR', 'QUERETARO', 'HIDALGO', 'COLIMA', 'CAMPECHE', 'AGUASCALIENTES'}


## SEMAR — formato pivote ancho (hoja DH)

Estructura: encabezados jerárquicos en filas 6-8 (estado → nivel de atención → mes), bloques de 6 filas por año desde la fila 9 (2017 en adelante). Cada estado tiene una o más columnas de "nivel de atención" (12 meses + TOTAL ANUAL); si el estado tiene más de un nivel, hay una columna `TOTAL` adicional que ya suma todos los niveles — si solo tiene un nivel, no existe esa columna y se usa directamente su `TOTAL ANUAL`.

In [13]:
_EXCLUIR_ESTADO_SEMAR = {
    "ENTIDAD FEDERATIVA", "TOTAL PRIMER NIVEL", "TOTAL SEGUNDO NIVEL",
    "TOTAL TERCER NIVEL", "TOTAL GLOBAL",
}

def _localizar_columnas_total_semar(df_raw: pd.DataFrame) -> dict[str, int]:
    """Por cada estado, ubica la columna que contiene su total anual agregado.

    Si el estado tiene más de un nivel de atención, usa la columna con
    nivel == 'TOTAL' (ya suma todos los niveles). Si solo tiene un nivel,
    no existe esa columna y se usa su única columna 'TOTAL ANUAL'.
    """
    fila_estado = df_raw.iloc[6].ffill()
    fila_nivel = df_raw.iloc[7]
    fila_mes = df_raw.iloc[8]

    estados = [e for e in fila_estado.dropna().unique() if e not in _EXCLUIR_ESTADO_SEMAR]

    col_total_por_estado = {}
    for estado in estados:
        cols = [c for c in range(df_raw.shape[1]) if fila_estado[c] == estado]
        col_nivel_total = [
            c for c in cols
            if isinstance(fila_nivel[c], str) and fila_nivel[c].strip().upper() == "TOTAL"
        ]
        if col_nivel_total:
            col_total_por_estado[estado] = col_nivel_total[0]
        else:
            cols_anual = [
                c for c in cols
                if isinstance(fila_mes[c], str) and fila_mes[c].strip().upper() == "TOTAL ANUAL"
            ]
            assert len(cols_anual) == 1, (estado, cols_anual)
            col_total_por_estado[estado] = cols_anual[0]

    return col_total_por_estado

In [14]:
_ANIO_MIN_SEMAR, _ANIO_MAX_SEMAR = 2019, 2023  # ventana comparable con IMSS/ISSSTE/IMSS Bienestar
_NORM_SEMAR = {
    "MICHOACAN": "MICHOACÁN",
    "YUCATAN": "YUCATÁN",
}
def _parse_semar(path, hoja: str = "DH") -> pd.DataFrame:
    raw = pd.read_excel(path, sheet_name=hoja, header=None)
    col_total_por_estado = _localizar_columnas_total_semar(raw)

    registros = []
    fila = 9  # primer bloque de año (2017)
    while fila < len(raw):
        anio = raw.iloc[fila, 0]
        if pd.isna(anio) or not isinstance(anio, (int, float)):
            break  # llegamos a la fila de TOTAL acumulado al final de la hoja
        anio = int(anio)
        if _ANIO_MIN_SEMAR <= anio <= _ANIO_MAX_SEMAR:
            for estado, col in col_total_por_estado.items():
                registros.append({
                    "estado": estado,
                    "anio": anio,
                    "surtidas": raw.iloc[fila, col],
                    "total": raw.iloc[fila + 3, col],
                })
        fila += 6

    df = pd.DataFrame(registros)
    df["estado"] = df["estado"].replace(_NORM_SEMAR)
    return df


df_semar = _parse_semar(DATA_DIR / "SEMAR_2017_ABRIL2024_ANEXO.xlsx", hoja="DH")
df_semar[["surtidas", "total"]] = df_semar[["surtidas", "total"]].round().astype(int)
print(df_semar.shape, df_semar["estado"].nunique(), sorted(df_semar.anio.unique()))
print(df_semar.groupby(["estado", "anio"])[["surtidas", "total"]].sum())

(90, 4) 18 [np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023)]
                      surtidas  total
estado          anio                 
BAJA CALIFORNIA 2019      1648   1703
                2020      1790   1849
                2021     11626  14503
                2022     12479  15377
                2023     13299  15339
...                        ...    ...
YUCATÁN         2019     10021  13638
                2020      7693   9463
                2021      7781  11107
                2022      8960  11249
                2023      9645  11117

[90 rows x 2 columns]


In [15]:
estados_base = set(pd.read_csv(DATA_DIR / "base_modelo_2019_2023.csv")["estado"].unique())
estados_semar = set(df_semar["estado"].unique())
print("En SEMAR pero no en base:", estados_semar - estados_base)
print("En base pero no en SEMAR:", estados_base - estados_semar)

En SEMAR pero no en base: set()
En base pero no en SEMAR: {'SAN LUIS POTOSI', 'PUEBLA', 'MÉXICO', 'TLAXCALA', 'NUEVO LEÓN', 'QUERETARO', 'DURANGO', 'ZACATECAS', 'AGUASCALIENTES', 'HIDALGO', 'GUANAJUATO', 'MORELOS', 'COAHUILA', 'CHIHUAHUA'}
